# MovieLens 100K LLM Rating Predictor
This notebook demonstrates using an LLM to predict user ratings.

In [ ]:
import pandas as pd
import os
import re
import time
from tqdm.notebook import tqdm
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

## 1. Setup Data Loader
We define functions to load `u.item`, `u.user`, `u1.base`, and `u1.test`.

In [ ]:
def load_data(data_dir="ml-100k"):
    # Load items (movies)
    item_cols = ['movie_id', 'title', 'release_date', 'video_release_date', 'imdb_url', 
                 'unknown', 'Action', 'Adventure', 'Animation', 'Childrens', 'Comedy', 
                 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 
                 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
    items_path = os.path.join(data_dir, 'u.item')
    items = pd.read_csv(items_path, sep='|', names=item_cols, encoding='latin-1', usecols=['movie_id', 'title', 'Action', 'Adventure', 'Animation', 'Childrens', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'])
    
    # Load users
    user_cols = ['user_id', 'age', 'gender', 'occupation', 'zip_code']
    users_path = os.path.join(data_dir, 'u.user')
    users = pd.read_csv(users_path, sep='|', names=user_cols, encoding='latin-1')
    
    # Load training ratings
    rating_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
    train_path = os.path.join(data_dir, 'u1.base')
    train_ratings = pd.read_csv(train_path, sep='\t', names=rating_cols)
    
    # Load test ratings
    test_path = os.path.join(data_dir, 'u1.test')
    test_ratings = pd.read_csv(test_path, sep='\t', names=rating_cols)
    
    return items, users, train_ratings, test_ratings

def get_user_history(user_id, train_ratings, items):
    user_ratings = train_ratings[train_ratings['user_id'] == user_id]
    user_ratings = pd.merge(user_ratings, items[['movie_id', 'title']], on='movie_id')
    
    high_rated = user_ratings[user_ratings['rating'] >= 4]['title'].tolist()
    low_rated = user_ratings[user_ratings['rating'] <= 2]['title'].tolist()
    
    high_rated_sample = high_rated[:15] if len(high_rated) > 15 else high_rated
    low_rated_sample = low_rated[:10] if len(low_rated) > 10 else low_rated
    
    return high_rated_sample, low_rated_sample

def get_movie_info(movie_id, items):
    movie = items[items['movie_id'] == movie_id].iloc[0]
    title = movie['title']
    
    genres = []
    genre_cols = ['Action', 'Adventure', 'Animation', 'Childrens', 'Comedy', 
                  'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 
                  'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
    for g in genre_cols:
        if movie[g] == 1:
            genres.append(g)
            
    genre_str = ", ".join(genres)
    return title, genre_str

def get_user_info(user_id, users):
    user = users[users['user_id'] == user_id].iloc[0]
    return user['age'], user['gender'], user['occupation']

## 2. Prompt Construction
We define a function to construct the prompt sent to the LLM.

In [ ]:
def generate_prompt(age, gender, occupation, high_rated_movies, low_rated_movies, target_movie, target_genres):
    prompt = f"""You are an advanced recommendation system and an expert movie critic.\nYou need to predict how a specific user will rate a target movie on a scale from 1 to 5.\n\n### User Profile\n- Age: {age}\n- Gender: {gender}\n- Occupation: {occupation}\n\n### User's Movie History\nThe user has previously ENJOYED these movies (rated 4 or 5 out of 5):\n{", ".join(high_rated_movies) if high_rated_movies else "None recorded."}\n\nThe user DISLIKED these movies (rated 1 or 2 out of 5):\n{", ".join(low_rated_movies) if low_rated_movies else "None recorded."}\n\n### Target Movie\n- Title: {target_movie}\n- Genres: {target_genres}\n\nPredict the rating this user will give to '{target_movie}'.\nOutput your prediction as a single integer between 1 and 5. Do not output any explanation or extra text.\n"""
    return prompt

## 3. LLM API Client
This uses either OpenAI, Gemini, or defaults to a dummy response (returning 3) if no keys are found.

In [ ]:
def get_rating_from_llm(prompt):
    prediction = None
    
    if "OPENAI_API_KEY" in os.environ:
        from openai import OpenAI
        client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        try:
            response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[
                    {"role": "system", "content": "You are a helpful assistant. Output ONLY a single integer."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.0,
                max_tokens=5
            )
            prediction = response.choices[0].message.content.strip()
        except Exception as e:
            print(f"OpenAI API Error: {e}")
            
    elif "GEMINI_API_KEY" in os.environ:
        import google.generativeai as genai
        genai.configure(api_key=os.environ["GEMINI_API_KEY"])
        try:
            model = genai.GenerativeModel('gemini-1.5-flash')
            response = model.generate_content(
                prompt,
                generation_config=genai.types.GenerationConfig(
                    temperature=0.0,
                    max_output_tokens=5,
                )
            )
            prediction = response.text.strip()
        except Exception as e:
            print(f"Gemini API Error: {e}")
    else:
        return 3

    try:
        match = re.search(r'\d', prediction)
        if match:
            rating = int(match.group())
            return max(1, min(5, rating))
        else:
            return 3
    except:
        return 3

## 4. Evaluation Loop
Run standard evaluation across a subset or the full test set.

In [ ]:
# Optional setup for API keys:
# os.environ['OPENAI_API_KEY'] = 'sk-...'
# os.environ['GEMINI_API_KEY'] = 'AIza...'

print("Loading data...")
items, users, train_ratings, test_ratings = load_data("ml-100k")
print(f"Loaded {len(train_ratings)} training ratings and {len(test_ratings)} test ratings.")

# Let's test on an arbitrary subset of size 10 to save time/API cost
subset_size = 10
test_df = test_ratings.sample(n=subset_size, random_state=42).reset_index(drop=True)

print(f"Evaluating {len(test_df)} user-movie pairs...")

actuals = []
predictions = []

for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    user_id = row['user_id']
    movie_id = row['movie_id']
    actual_rating = row['rating']
    
    try:
        age, gender, occupation = get_user_info(user_id, users)
        high_rated, low_rated = get_user_history(user_id, train_ratings, items)
        target_title, target_genres = get_movie_info(movie_id, items)
        
        prompt = generate_prompt(
            age=age,
            gender=gender,
            occupation=occupation,
            high_rated_movies=high_rated,
            low_rated_movies=low_rated,
            target_movie=target_title,
            target_genres=target_genres
        )
        
        predicted_rating = get_rating_from_llm(prompt)
        
        actuals.append(actual_rating)
        predictions.append(predicted_rating)
    except Exception as e:
        print(f"Error predicting for user {user_id} and movie {movie_id}: {e}")
        actuals.append(actual_rating)
        predictions.append(3)

test_df['predicted_rating'] = predictions

rmse = root_mean_squared_error(actuals, predictions)
mae = mean_absolute_error(actuals, predictions)

print("\n--- Final Results ---")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
